> **UNQUANTIZED RERUN** — generated copy of `23_e3_beam_negative.ipynb` with
> `meta-llama/Llama-3.1-8B-Instruct` (bf16, sharded via `device_map="auto"`)
> in place of the AWQ-INT4 variant. All `resources/` outputs carry the
> `_unquantized` suffix; the pair selection (`15_e0_selected_pairs.json`) is
> shared with the quantized runs so results are directly comparable.
> Hardcoded reference values quoted in comments/markdown still describe the
> AWQ runs. Original notebook left untouched.


# 23 — E3b: negative / mixed steering under the final recipe (beam + λ)

`22_e3_negative_steering.ipynb` ran the negative-steering comparison in the greedy tier and found: suppression works (Δrepel −0.8…−1.5, far beyond attract-only's incidental level) **but bleeds into the attracted concept** (Δattract often < 0; clean rates 0.05–0.60), with strong direction asymmetries and the high-ρ twin problem.

This notebook re-runs the same design under the E1b **final recipe** — true beam search, λ ∈ {0, 0.4} — asking:

1. Does beam's stronger search fix the **collateral-suppression** problem (raise Δattract while keeping Δrepel < 0)?
2. Does the fluency dial behave for negative steering as it did for positive (λ=0.4 ≈ guardrail at ~85% effect)?
3. Do the greedy-tier direction asymmetries and the high-ρ twin failure survive?

Grid: 3 methods (attract-only control, frame-diff, score-neg) × 2 λ × 2 directions × 3 strata = 36 cells, 720 records. Same prompts/baselines as E1/E3.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

from frames.evaluation.harness import EvaluationHarness
from frames.representations import FrameUnembeddingRepresentation

In [ ]:
fur = FrameUnembeddingRepresentation.from_model_id(
    "meta-llama/Llama-3.1-8B-Instruct",
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

MIN_LEMMAS = 11
MAX_TOKENS = 3
K = 4
STEPS = 24
LAMBDAS = (0.0, 0.4)

SELECTION = json.loads(Path("resources/15_e0_selected_pairs.json").read_text())[
    "selected"
]
for stratum, info in SELECTION.items():
    print(f"{stratum:>6}: {info['a']} / {info['b']}  rho={info['rho']:.3f}")

In [ ]:
def chat(user: str, assistant: str = "") -> str:
    return (
        f"<|start_header_id|>user<|end_header_id|>{user}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>{assistant}"
    )


PROMPTS = [
    chat("What men can be?", "1."),
    chat("What women can be?", "1."),
    chat("Tell me a short story.", "Once"),
    chat("Describe your favorite place.", "My favorite place is"),
    chat("What do you like to do on weekends?", "I like"),
    chat("Give me some advice for a good life.", "1."),
    chat(
        "What is the most important thing in the world?", "The most important thing is"
    ),
    chat("Describe the weather today.", "Today the weather is"),
    chat("What should I cook for dinner?", "You should"),
    chat("Name things that make people happy.", "1."),
    chat("What did you do today?", "Today I"),
    chat("Describe a memorable meal.", "The most memorable meal"),
    chat("What is a good gift idea?", "A good gift"),
    chat("Describe your hometown.", "My hometown is"),
    chat("Tell me about your favorite season.", "My favorite season is"),
    chat("What makes a good friend?", "A good friend"),
    chat("Describe a place you would like to visit.", "I would like to visit"),
    chat("What do you do to relax?", "To relax, I"),
    chat("Tell me something interesting.", "Here is something interesting:"),
    chat("Describe your morning routine.", "My morning starts"),
]

LOG_PATH = Path("resources/23_e3_beam_negative_unquantized.jsonl")
LOG_PATH.unlink(missing_ok=True)

harness = EvaluationHarness(
    fur, LOG_PATH, min_lemmas_per_synset=MIN_LEMMAS, max_token_count=MAX_TOKENS
)
baseline_texts = harness.generate_baseline(PROMPTS, max_new_tokens=STEPS + 2)
print("baseline ready")

In [ ]:
def make_methods(attract, repel, lam: float) -> dict:
    differential = attract - repel
    beam = fur.generate_with_topk_beam_guide
    return {
        "attract-only": lambda: beam(
            PROMPTS,
            concepts=[attract],
            k=K,
            steps=STEPS,
            fluency_weight=lam,
            batch_size=4,
        ),
        "frame-diff": lambda: beam(
            PROMPTS,
            concepts=[differential],
            k=K,
            steps=STEPS,
            fluency_weight=lam,
            batch_size=4,
        ),
        "score-neg": lambda: beam(
            PROMPTS,
            concepts=[attract, repel],
            weights=[1.0, -1.0],
            normalize="zscore",
            k=K,
            steps=STEPS,
            fluency_weight=lam,
            batch_size=4,
        ),
    }

In [ ]:
for stratum, info in SELECTION.items():
    concept = {
        name: fur.get_concept_cached(name, MIN_LEMMAS, MAX_TOKENS)
        for name in (info["a"], info["b"])
    }

    for attract_name, repel_name in ((info["a"], info["b"]), (info["b"], info["a"])):
        for lam in LAMBDAS:
            methods = make_methods(concept[attract_name], concept[repel_name], lam)
            for method, generate in methods.items():
                torch.cuda.empty_cache()
                texts, probe = generate()
                harness.evaluate(
                    PROMPTS,
                    texts,
                    [attract_name, repel_name],
                    config={
                        "stratum": stratum,
                        "rho": info["rho"],
                        "method": method,
                        "lam": lam,
                        "attract": attract_name,
                        "repel": repel_name,
                        "k": K,
                        "steps": STEPS,
                    },
                    probe=probe,
                    baseline_texts=baseline_texts,
                    concepts=concept,
                )
                print(
                    f"{stratum} / +{attract_name} -{repel_name} / {method} / lam={lam:g}: done"
                )

In [ ]:
parsed = [json.loads(line) for line in LOG_PATH.read_text().strip().split("\n")]
assert len(parsed) == 3 * 2 * 3 * len(LAMBDAS) * len(PROMPTS)
assert all(r["expression"] is not None for r in parsed)
print(f"gate: {len(parsed)} records parse, expression logged")

In [ ]:
rows = []
for record in parsed:
    config = record["config"]
    expr = record["expression"]
    rows.append(
        {
            "stratum": config["stratum"],
            "rho": config["rho"],
            "method": config["method"],
            "lam": config["lam"],
            "direction": f"+{config['attract'].split('.')[0]}"
            f"-{config['repel'].split('.')[0]}",
            "delta_attract": expr[config["attract"]]["delta"],
            "delta_repel": expr[config["repel"]]["delta"],
            "net": expr[config["attract"]]["delta"] - expr[config["repel"]]["delta"],
            "clean": expr[config["attract"]]["delta"] > 0
            and expr[config["repel"]]["delta"] < 0,
            "ppl_ratio": record["ppl_ratio"],
            "fluency_flag": record["fluency_flag"],
        }
    )
results = pd.DataFrame(rows)

summary = (
    results.groupby(["stratum", "rho", "direction", "method", "lam"])
    .agg(
        delta_attract=("delta_attract", "mean"),
        delta_repel=("delta_repel", "mean"),
        net=("net", "mean"),
        clean_rate=("clean", "mean"),
        ppl_ratio=("ppl_ratio", "mean"),
        flag_share=("fluency_flag", "mean"),
    )
    .reset_index()
    .sort_values(
        ["rho", "direction", "lam", "net"], ascending=[True, True, True, False]
    )
)
summary.to_csv("resources/23_e3_beam_summary_unquantized.csv", index=False)
summary.round(4)

## Quadrant view — beam tier vs the greedy tier

Target quadrant is top-left (attract up, repel suppressed). Faded markers are the greedy-tier results from `22_e3_negative_steering` for the same cells; arrows would all point up if beam fixes collateral suppression.

In [ ]:
greedy = pd.read_csv("resources/22_e3_summary_unquantized.csv")

method_colors = {
    "attract-only": "tab:gray",
    "frame-diff": "tab:blue",
    "score-neg": "tab:orange",
}
fig, axes = plt.subplots(2, 3, figsize=(15, 9), sharex=True, sharey=True)

for row_axes, lam in zip(axes, LAMBDAS):
    for ax, (stratum, info) in zip(row_axes, SELECTION.items()):
        old = greedy[greedy["stratum"] == stratum]
        for _, r in old.iterrows():
            ax.scatter(
                r["delta_repel"],
                r["delta_attract"],
                s=40,
                color=method_colors[r["method"]],
                alpha=0.25,
                marker="o",
            )
        cell = summary[(summary["stratum"] == stratum) & (summary["lam"] == lam)]
        for _, r in cell.iterrows():
            ax.scatter(
                r["delta_repel"],
                r["delta_attract"],
                s=90,
                color=method_colors[r["method"]],
                edgecolor="black",
                linewidth=0.8,
            )
            ax.annotate(
                f"{r['method']}\n{r['direction']}",
                (r["delta_repel"], r["delta_attract"]),
                textcoords="offset points",
                xytext=(6, 2),
                fontsize=6,
            )
        ax.axhline(0, color="gray", lw=0.8)
        ax.axvline(0, color="gray", lw=0.8)
        ax.set_title(f"{stratum} (rho={info['rho']:.2f}), lam={lam:g}")
for ax in axes[-1]:
    ax.set_xlabel("delta expr (repelled)  <- better")
for row_axes in axes:
    row_axes[0].set_ylabel("delta expr (attracted)")
handles = [
    plt.Line2D([], [], marker="o", ls="", color=c, label=m)
    for m, c in method_colors.items()
]
axes[0, -1].legend(handles=handles, fontsize=8, loc="lower right")
plt.tight_layout()
plt.savefig("resources/23_e3_beam_quadrants_unquantized.png", dpi=150)
plt.show()

In [ ]:
print("clean rate (attract > 0 AND repel < 0) by tier:")
greedy_clean = greedy.groupby("method")["clean_rate"].mean()
for lam in LAMBDAS:
    beam_clean = summary[summary["lam"] == lam].groupby("method")["clean_rate"].mean()
    table = pd.DataFrame({"greedy": greedy_clean, f"beam lam={lam:g}": beam_clean})
    print(table.round(3), "\n")

print("mean PPL ratio by tier:")
for lam in LAMBDAS:
    beam_ppl = summary[summary["lam"] == lam].groupby("method")["ppl_ratio"].mean()
    print(f"beam lam={lam:g}:")
    print(beam_ppl.round(2), "\n")
print("greedy:")
print(greedy.groupby("method")["ppl_ratio"].mean().round(2))

In [ ]:
def answer(text: str) -> str:
    return text.split("assistant<|end_header_id|>")[-1]


for stratum in SELECTION:
    for method in ("frame-diff", "score-neg"):
        for lam in LAMBDAS:
            example = next(
                r
                for r in parsed
                if r["config"]["stratum"] == stratum
                and r["config"]["method"] == method
                and r["config"]["lam"] == lam
                and "short story" in r["prompt"]
            )
            config = example["config"]
            deltas = {
                name.split(".")[0]: round(v["delta"], 3)
                for name, v in example["expression"].items()
            }
            print(
                f"[{stratum} / +{config['attract'].split('.')[0]}"
                f"-{config['repel'].split('.')[0]} / {method} / lam={lam:g}] deltas={deltas}"
            )
            print("   ", answer(example["text"])[:125])
    print()